In [5]:
%pip install cryptography

import os, json, base64, hashlib, hmac, secrets, sqlite3, threading
from datetime import datetime, timezone
from cryptography.fernet import Fernet
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

DEMO_MODE = os.environ.get("IIOT_DEMO_MODE", "1").strip().lower() in {"1", "true", "yes", "on"}
DB_URI = False
_DB_KEEPER = None
if DEMO_MODE:
    fernet_raw = Fernet.generate_key().decode("ascii")
    admin_password = secrets.token_urlsafe(24)
    DB_PATH = f"file:iiot_security_demo_{secrets.token_hex(8)}?mode=memory&cache=shared"
    DB_URI = True
    _DB_KEEPER = sqlite3.connect(DB_PATH, uri=True, check_same_thread=False)
    print("Demo mode: using ephemeral credentials and an in-memory registry.")
else:
    fernet_raw = os.environ.get("IIOT_REGISTRY_FERNET_KEY")
    admin_password = os.environ.get("IIOT_ADMIN_PASSWORD")
    missing_settings = [name for name, value in (("IIOT_REGISTRY_FERNET_KEY", fernet_raw), ("IIOT_ADMIN_PASSWORD", admin_password)) if not value]
    if missing_settings:
        raise RuntimeError("Configure these environment variables before persistent mode: " + ", ".join(missing_settings) + ". Do not hard-code secrets in the notebook.")
    if len(admin_password) < 12:
        raise RuntimeError("Set a strong IIOT_ADMIN_PASSWORD (12+ characters).")
    DB_PATH = os.environ.get("IIOT_DB_PATH", "iiot_security.sqlite3")
registry_cipher = Fernet(fernet_raw.encode("ascii"))
DB_LOCK = threading.Lock()
MAX_CLOCK_SKEW_SECONDS = 120

def db():
    conn = sqlite3.connect(DB_PATH, timeout=10, uri=DB_URI)
    conn.row_factory = sqlite3.Row
    return conn

with db() as conn:
    conn.executescript("""
    CREATE TABLE IF NOT EXISTS devices(device_id TEXT PRIMARY KEY, key_ct BLOB NOT NULL, enabled INTEGER NOT NULL DEFAULT 1, last_seq INTEGER NOT NULL DEFAULT -1);
    CREATE TABLE IF NOT EXISTS users(username TEXT PRIMARY KEY, salt BLOB NOT NULL, pw_hash BLOB NOT NULL, role TEXT NOT NULL, enabled INTEGER NOT NULL DEFAULT 1);
    CREATE TABLE IF NOT EXISTS audit(id INTEGER PRIMARY KEY, at TEXT NOT NULL, subject TEXT, event TEXT NOT NULL, detail TEXT);
    """)

def audit(subject, event, detail=""):
    with DB_LOCK, db() as conn:
        conn.execute("INSERT INTO audit(at,subject,event,detail) VALUES(?,?,?,?)", (datetime.now(timezone.utc).isoformat(), subject, event, detail[:300]))
print("Local security registry initialized.")

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Demo mode: using ephemeral credentials and an in-memory registry.
Local security registry initialized.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\iamwa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
def provision_device(device_id):
    created = False
    with DB_LOCK, db() as conn:
        row = conn.execute("SELECT key_ct FROM devices WHERE device_id=?", (device_id,)).fetchone()
        if row is None:
            key = AESGCM.generate_key(bit_length=256)
            encrypted = registry_cipher.encrypt(key)
            conn.execute("INSERT INTO devices(device_id,key_ct) VALUES(?,?)", (device_id, encrypted))
            created = True
        else:
            key = registry_cipher.decrypt(row["key_ct"])
    if created:
        audit(device_id, "device_provisioned")
    return base64.urlsafe_b64encode(key).decode("ascii")

def set_device_enabled(device_id, enabled):
    with DB_LOCK, db() as conn:
        cur = conn.execute("UPDATE devices SET enabled=? WHERE device_id=?", (int(enabled), device_id))
        if cur.rowcount != 1: raise KeyError("Unknown device")
    audit(device_id, "device_enabled" if enabled else "device_disabled")

DEVICE_ID = "sensor-001"
DEVICE_KEY_B64 = provision_device(DEVICE_ID)
assert provision_device(DEVICE_ID) == DEVICE_KEY_B64
print(f"Provisioned {DEVICE_ID}; securely deliver its key to the device.")

Provisioned sensor-001; securely deliver its key to the device.


In [7]:
def header_bytes(device_id, sequence, timestamp):
    return json.dumps({"device_id":device_id,"sequence":int(sequence),"timestamp":timestamp}, sort_keys=True, separators=(",", ":")).encode()

def make_envelope(device_id, key_b64, sequence, measurements):
    key = base64.urlsafe_b64decode(key_b64)
    timestamp = datetime.now(timezone.utc).isoformat(timespec="seconds")
    nonce = secrets.token_bytes(12)
    plaintext = json.dumps(measurements, sort_keys=True, separators=(",", ":")).encode()
    ciphertext = AESGCM(key).encrypt(nonce, plaintext, header_bytes(device_id, sequence, timestamp))
    return {"device_id":device_id,"sequence":sequence,"timestamp":timestamp,
            "nonce":base64.b64encode(nonce).decode(),"ciphertext":base64.b64encode(ciphertext).decode()}

class MessageRejected(Exception): pass

def receive_envelope(msg):
    try:
        device_id = str(msg["device_id"])
        seq = int(msg["sequence"])
        timestamp = str(msg["timestamp"])
        parsed = datetime.fromisoformat(timestamp.replace("Z", "+00:00"))
        if parsed.tzinfo is None: raise ValueError("timezone missing")
        nonce = base64.b64decode(msg["nonce"], validate=True)
        ciphertext = base64.b64decode(msg["ciphertext"], validate=True)
    except (KeyError, TypeError, ValueError) as exc:
        raise MessageRejected(f"Malformed envelope: {exc}") from exc
    if seq < 0 or len(nonce) != 12: raise MessageRejected("Invalid sequence or nonce")
    if abs((datetime.now(timezone.utc)-parsed.astimezone(timezone.utc)).total_seconds()) > MAX_CLOCK_SKEW_SECONDS:
        audit(device_id, "stale_message_rejected")
        raise MessageRejected("Timestamp outside allowed clock skew")
    with DB_LOCK:
        conn = db()
        try:
            conn.execute("BEGIN IMMEDIATE")
            row = conn.execute("SELECT key_ct,enabled,last_seq FROM devices WHERE device_id=?", (device_id,)).fetchone()
            if row is None or not row["enabled"]: raise MessageRejected("Device is untrusted or disabled")
            key = registry_cipher.decrypt(row["key_ct"])
            try:
                clear = AESGCM(key).decrypt(nonce, ciphertext, header_bytes(device_id,seq,timestamp))
            except Exception as exc:
                raise MessageRejected("Authentication failed: tampering or wrong key") from exc
            if seq <= row["last_seq"]: raise MessageRejected("Replay detected: sequence is not newer")
            data = json.loads(clear.decode())
            if not isinstance(data, dict): raise MessageRejected("Payload must be a JSON object")
            conn.execute("UPDATE devices SET last_seq=? WHERE device_id=?", (seq,device_id))
            conn.commit()
        except Exception:
            conn.rollback()
            raise
        finally: conn.close()
    audit(device_id, "message_accepted", f"sequence={seq}")
    return {"device_id":device_id,"sequence":seq,"measurements":data,"timestamp":timestamp}

envelope = make_envelope(DEVICE_ID, DEVICE_KEY_B64, 1, {"temperature_c":26.4,"humidity_pct":48.2})
receive_envelope(envelope)

{'device_id': 'sensor-001',
 'sequence': 1,
 'measurements': {'humidity_pct': 48.2, 'temperature_c': 26.4},
 'timestamp': '2026-11-17T18:29:48+00:00'}

In [8]:
ROLES = {"admin", "operator", "viewer"}
def password_hash(password, salt=None):
    if not isinstance(password,str) or len(password)<12: raise ValueError("Password must be at least 12 characters")
    salt = salt or secrets.token_bytes(16)
    digest = hashlib.scrypt(password.encode(), salt=salt, n=2**14, r=8, p=1, dklen=32)
    return salt,digest

def create_user(username,password,role):
    if role not in ROLES: raise ValueError("Unknown role")
    salt,digest=password_hash(password)
    with DB_LOCK, db() as conn:
        conn.execute("INSERT INTO users(username,salt,pw_hash,role) VALUES(?,?,?,?)",(username,salt,digest,role))
    audit(username,"user_created",role)

def authenticate(username,password):
    with db() as conn: row=conn.execute("SELECT salt,pw_hash,role,enabled FROM users WHERE username=?",(username,)).fetchone()
    if row is None or not row["enabled"]: return None
    if not isinstance(password, str):
        audit(username, "login_failed")
        return None
    candidate = hashlib.scrypt(password.encode(), salt=row["salt"], n=2**14, r=8, p=1, dklen=32)
    if not hmac.compare_digest(candidate,row["pw_hash"]):
        audit(username,"login_failed"); return None
    audit(username,"login_succeeded")
    return {"username":username,"role":row["role"]}

with db() as conn: exists=conn.execute("SELECT 1 FROM users WHERE username='admin'").fetchone()
if not exists: create_user("admin",admin_password,"admin")
print("Admin account ready; existing password is not overwritten.")
assert authenticate("admin", "short") is None
print("Short-password login is rejected safely.")

Admin account ready; existing password is not overwritten.
Short-password login is rejected safely.


In [9]:
PERMISSIONS = {
    "viewer": {"telemetry:read"},
    "operator": {"telemetry:read", "actuator:control"},
    "admin": {"telemetry:read", "actuator:control", "device:manage", "user:manage"},
}
def authorize(user, permission):
    ok = bool(user and permission in PERMISSIONS.get(user.get("role"), set()))
    audit(user.get("username") if user else None, "authorization_allowed" if ok else "authorization_denied", permission)
    return ok

def require_permission(user, permission):
    if not authorize(user, permission): raise PermissionError(f"Permission required: {permission}")

admin = authenticate("admin", admin_password)
print("Admin can manage devices:", authorize(admin,"device:manage"))

Admin can manage devices: True


In [10]:
try:
    receive_envelope(envelope)  # same sequence was already accepted
except MessageRejected as exc:
    print("Replay rejected:", exc)

fresh = make_envelope(DEVICE_ID, DEVICE_KEY_B64, 2, {"temperature_c":27.0})
tampered = dict(fresh)
tampered["ciphertext"] = base64.b64encode(b"modified").decode()
try:
    receive_envelope(tampered)
except MessageRejected as exc:
    print("Tampering rejected:", exc)

Replay rejected: Replay detected: sequence is not newer
Tampering rejected: Authentication failed: tampering or wrong key
